In [23]:
import numpy as np
import random

In [24]:
def collapse(ket):
    p = ket.copy()

    p = p * np.conjugate(p) # Performs |ket|^2 which gets turns p into a probability distribution
    args = np.arange(len(ket))
    dex = np.random.choice(args, p=p/p.sum()) # Normalization on p in case of numerical stability error (should sum to 1)

    new_ket = np.zeros_like(ket)
    new_ket[dex] = 1

    return new_ket

In [25]:
change_to_x_basis = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]])
zero_ket = np.array([1,0])
one_ket = np.array([0,1])

plus_ket = zero_ket @ change_to_x_basis
minus_ket = one_ket @ change_to_x_basis

ket_dict = {
    (0, 'Z'): zero_ket,
    (1, 'Z'): one_ket,
    (0, 'X'): plus_ket,
    (1, 'X'): minus_ket
}

In [26]:
# Create the Message to be sent
n = 100
message = ''

# String of Basis's that Alice chose
alice_basis = ''
encrypted_message = []

for _ in range(n):
    bit = random.randint(0, 1)
    basis = random.choice(['Z', 'X'])

    encrypted_message.append(ket_dict.get((bit, basis)))
    alice_basis += basis
    
    message += f'{bit}'

In [27]:
boo = True # If true Eve is present

if boo:
    for i in range(n):
        ket = encrypted_message[i]
        basis = random.choice(['Z', 'X'])
    
        if basis == 'X':
            ket = change_to_x_basis @ ket
        encrypted_message[i] = ket

In [28]:
decrypted_message = ''
bob_basis = ''

for i in range(n):
    ket = encrypted_message[i]
    basis = random.choice(['Z', 'X'])

    if basis == 'X': # Following operation works as X is orthogonal
        ket = change_to_x_basis @ ket
    decrypted_message += f'{np.argmax(collapse(ket))}'
    bob_basis += basis

In [29]:
h = n // 2
secret_key = ''
accuracy = h

# acc_A and acc_B are shared publicly to compare, this is to see if anyone was listening
acc_A = alice_basis[:h]
sec_A = alice_basis[h:]

acc_B = bob_basis[:h]
sec_B = bob_basis[h:]

for i in range(h):
    if sec_A[i] == sec_B[i]:
        secret_key += message[i + h]
    if acc_A[i] == acc_B[i]:
        if message[i] != decrypted_message[i]:
            accuracy -= 1

print(secret_key)
print(f'Accuracy: {round(accuracy / h, 4) * 100}%')

0101000000010111010010111100
Accuracy: 90.0%
